## Subjective-value MVPA sandbox

This notebook walks through a first-pass subjective-value prediction run for a single subject using the shared MVPA data layer.

In [1]:
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

from dd_kable_analysis.config_loader import load_config
from dd_kable_analysis.mvpa import (
    build_subject_behav_bold_df,
    decode_subject_atlas_rois,
)


In [2]:
cfg = load_config()
sub_list_path = cfg.subject_lists / 'mvpa_subject_list.txt'

sub_ids = [
    line.strip() for line in sub_list_path.read_text().splitlines() if line.strip()
]
first_sub_id = sorted(sub_ids)[0]
first_sub_id

'dmp0011'

In [3]:
atlas_path = (
    Path(cfg.masks_dir)
    / 'pauli_subcort_rois_neurovault_collection_3145'
    / 'pauli_roi_atlas_thr0.50_to-groupmask.nii.gz'
)

atlas_img = nib.load(str(atlas_path))
roi_labels = {
    int(label)
    for label in np.unique(atlas_img.get_fdata().astype(int))
    if int(label) != 0
}

print(f'Using subject: {first_sub_id}')
print(f'Atlas: {atlas_path}')
print(f'Number of ROIs: {len(roi_labels)}')
print(sorted(roi_labels))

Using subject: dmp0011
Atlas: /oak/stanford/groups/russpold/users/buckholtz/DD_Kable/derivatives/masks/pauli_subcort_rois_neurovault_collection_3145/pauli_roi_atlas_thr0.50_to-groupmask.nii.gz
Number of ROIs: 16
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]


In [4]:
out = build_subject_behav_bold_df(
    cfg,
    sub_id=first_sub_id,
    y_col='SV_LL',
    verbose=True,
    strict=True,
)

[dmp0011] QA-passed runs: ['1', '2', '3', '4']
[dmp0011] trial regressors in design (pre-filter): 116
[dmp0011] omitted high-VIF trials: 0
[dmp0011] missing beta files: 0
[dmp0011] missing subjective-value rows after merge: 0
[dmp0011] kept trials (post-filter): 116
[dmp0011] kept by run: {'1': 29, '2': 29, '3': 29, '4': 29}
[dmp0011] runs with >= 20 kept trials: ['1', '2', '3', '4']


In [11]:
roi_summary_df, trialwise_df = decode_subject_atlas_rois(
    cfg,
    sub_id=first_sub_id,
    atlas_img=str(atlas_path),
    y_col='SV_LL',
    min_voxels_per_roi=50,
    small_thr=1e-4,
    max_small_frac=0.05,
    require_all_runs=True,
    verbose=True,
    return_trialwise=True,
    trialwise_rois=roi_labels,
)


[dmp0011] QA-passed runs: ['1', '2', '3', '4']
[dmp0011] trial regressors in design (pre-filter): 116
[dmp0011] omitted high-VIF trials: 0
[dmp0011] missing beta files: 0
[dmp0011] missing subjective-value rows after merge: 0
[dmp0011] kept trials (post-filter): 116
[dmp0011] kept by run: {'1': 29, '2': 29, '3': 29, '4': 29}
[dmp0011] runs with >= 20 kept trials: ['1', '2', '3', '4']
[global] X_all=(116, 240143) (mask vox=240143) y=(116,) groups=(116,)
[global] trials per run:
 1    29
2    29
3    29
4    29
[roi map] n_rois=16
[roi map] voxels per ROI percentiles: [   2.    8.   54. 1463. 1655.]


In [12]:
print('QC summary:')
print(f'  good runs: {out.good_runs}')
print(f'  trials kept by run: {out.trials_kept_by_run}')
print(f'  runs passing trial threshold: {out.runs_passing_trial_threshold}')
print(f'  n_trials_after_vif_and_missing: {out.n_trials_after_vif_and_missing}')
print()
print('ROI summary head:')
display(roi_summary_df.head())
print()
print('Trialwise predictions head:')
display(trialwise_df.head())

QC summary:
  good runs: ['1', '2', '3', '4']
  trials kept by run: {'1': 29, '2': 29, '3': 29, '4': 29}
  runs passing trial threshold: ['1', '2', '3', '4']
  n_trials_after_vif_and_missing: 116

ROI summary head:


,sub_id,roi_label,n_trials,n_runs,n_vox_preQC,n_vox_postQC,r,r2_cv,fisher_z,rmse,mean_alpha
0,dmp0011,1,116,4,1655,1641,-0.195117,-0.053839,-0.197652,10.032145,751961.899926
1,dmp0011,2,116,4,1399,1369,-0.112414,-0.043095,-0.112891,9.980872,505916.933063
2,dmp0011,3,116,4,122,112,0.023875,0.008060,0.023880,9.733059,250496.276093
3,dmp0011,14,116,4,183,120,-0.017680,-0.008283,-0.017682,9.812911,250496.276093



Trialwise predictions head:


,sub_id,roi_label,run,trial_type,Delay,amount,choseAccept,y,yhat_oos
0,dmp0011,1,1,trial00,58,66,1,29.118120,12.391465
1,dmp0011,1,1,trial01,95,39,0,12.684361,14.442231
2,dmp0011,1,1,trial02,172,69,0,14.507342,13.983769
3,dmp0011,1,1,trial03,42,23,0,11.996570,16.227545
4,dmp0011,1,1,trial04,142,77,1,18.775637,15.997893


In [13]:
roi_summary_df

,sub_id,roi_label,n_trials,n_runs,n_vox_preQC,n_vox_postQC,r,r2_cv,fisher_z,rmse,mean_alpha
0,dmp0011,1,116,4,1655,1641,-0.195117,-0.053839,-0.197652,10.032145,751961.899926
1,dmp0011,2,116,4,1399,1369,-0.112414,-0.043095,-0.112891,9.980872,505916.933063
2,dmp0011,3,116,4,122,112,0.023875,0.008060,0.023880,9.733059,250496.276093
3,dmp0011,14,116,4,183,120,-0.017680,-0.008283,-0.017682,9.812911,250496.276093
